# CareLoop Clinics — Part 3: No-Show Risk Model
This notebook completes Tasks 3.1–3.5. Model and threshold selection use training data only.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

## Task 3.1 — Features, split and baseline

In [ ]:
df = pd.read_csv("appointments_clean.csv")
X = df[["age","distance_km","fee_inr","lead_days","prior_no_shows","reminder_sent","booking_channel","visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel","visit_type"])
y = df["no_show"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Features:", X.shape[1], list(X.columns))
print("Train/test:", len(X_train), len(X_test))
print("Test no-shows:", y_test.sum())
print("Always-Attended baseline:", (y_test == 0).mean())

## Task 3.2 — Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("F1:", f1_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))
coef_table = pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_[0], "exp_coefficient": np.exp(lr.coef_[0])})
print(coef_table.sort_values("coefficient"))
idx = list(X.columns).index("reminder_sent")
scaled_or = np.exp(lr.coef_[0][idx])
real_change_or = np.exp(lr.coef_[0][idx] / scaler.scale_[idx])
print("Reminder exp(coef):", scaled_or)
print("Reminder SD:", scaler.scale_[idx])
print("Reminder No-to-Yes odds ratio:", real_change_or)
print("Interpretation: in this model, holding other features constant, reminders are associated with lower no-show odds; this is not causal proof.")

## Tasks 3.3 and 3.4 — Trees and random forest

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print("Full tree train/test/depth:", full_tree.score(X_train,y_train), full_tree.score(X_test,y_test), full_tree.get_depth())
print("Depth-3 tree train/test:", tree3.score(X_train,y_train), tree3.score(X_test,y_test))
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test); rf_prob = rf.predict_proba(X_test)[:,1]
print("RF train/test:", rf.score(X_train,y_train), accuracy_score(y_test,rf_pred))
print("RF recall/AUC:", recall_score(y_test,rf_pred), roc_auc_score(y_test,rf_prob))
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3))
comparison = pd.DataFrame([
 {"model":"Logistic regression","accuracy":accuracy_score(y_test,lr_pred),"recall":recall_score(y_test,lr_pred),"roc_auc":roc_auc_score(y_test,lr_prob)},
 {"model":"Depth-3 tree","accuracy":accuracy_score(y_test,tree3.predict(X_test)),"recall":recall_score(y_test,tree3.predict(X_test)),"roc_auc":roc_auc_score(y_test,tree3.predict_proba(X_test)[:,1])},
 {"model":"Random forest","accuracy":accuracy_score(y_test,rf_pred),"recall":recall_score(y_test,rf_pred),"roc_auc":roc_auc_score(y_test,rf_prob)}])
print(comparison)

## Task 3.5 — Tune, choose honestly, select threshold, then test once

In [ ]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators":[100,200],"max_depth":[3,5,8]}, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("Best tuned RF params:", grid.best_params_)
print("Best tuned RF CV AUC:", grid.best_score_)
lr_pipe = Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=1000))])
models = {
 "logistic_regression": lr_pipe,
 "depth3_tree": DecisionTreeClassifier(max_depth=3, random_state=42),
 "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
 "tuned_forest": grid.best_estimator_}
cv_auc = {
 "logistic_regression": cross_val_score(lr_pipe,X_train,y_train,cv=5,scoring="roc_auc").mean(),
 "depth3_tree": cross_val_score(models["depth3_tree"],X_train,y_train,cv=5,scoring="roc_auc").mean(),
 "random_forest": cross_val_score(models["random_forest"],X_train,y_train,cv=5,scoring="roc_auc").mean(),
 "tuned_forest": grid.best_score_}
print("Training CV AUC:", cv_auc)
winner_name = max(cv_auc, key=cv_auc.get); winner = models[winner_name]
print("Winner:", winner_name)
oof = cross_val_predict(winner,X_train,y_train,cv=5,method="predict_proba")[:,1]
threshold_rows=[]
for threshold in [0.50,0.35]:
    pred=(oof>=threshold).astype(int)
    threshold_rows.append({"threshold":threshold,"precision":precision_score(y_train,pred),
                           "recall":recall_score(y_train,pred),"f1":f1_score(y_train,pred),"flagged":int(pred.sum())})
threshold_table=pd.DataFrame(threshold_rows)
print(threshold_table)
chosen_threshold=float(threshold_table.loc[threshold_table["f1"].idxmax(),"threshold"])
print("Chosen threshold:", chosen_threshold)
winner.fit(X_train,y_train)
probs=winner.predict_proba(X_test)[:,1]
flags=(probs>=chosen_threshold).astype(int)
print("Final test precision:",precision_score(y_test,flags))
print("Final test recall:",recall_score(y_test,flags))
print("Final test flagged:",int(flags.sum()))
print("Final test ROC-AUC:",roc_auc_score(y_test,probs))
test_scores=pd.DataFrame({"appointment_id":df.loc[X_test.index,"appointment_id"].values,
                          "no_show":y_test.values,"risk_prob":probs,"flag":flags})
test_scores.to_csv("test_risk_scores.csv",index=False)
print(test_scores.head(), test_scores.shape)
print("A real clinic would choose the threshold by balancing evening call capacity against the cost of an empty slot.")